#### Files of Interest
data/constructed_datasets/gss.csv

#### Document missing values for ONET_SOC_CODE.
Identify missing values and outliers in the merged dataset and check whether they are random or follow a pattern (ex. which ISCO-2008 or OCC10 codes are missing). Document patterns and use this to determine how best to handle missing values.

#### Identify which variables are of interest.
One of the goals of the project is to find the measure “job fit” based on a person’s job values compared to their current jobs. Take a look at any documentation available regarding your GSS or ISSP and document which variables are of interest (ex. “importance of job security”).
NOTE: JOB SATISFACTION VARIABLE IS VERY IMPORTANT, also take a look at demographic data, and any other job data that might not be available from O*NET (ex. Time spent at work)

#### Document missing values and outliers
After identifying variables of interest, explore how many/which responses are missing our variables of interest and document your findings (how many rows are actually usable out of the total size, what strategies can we use to address them, etc.)

#### Initial EDA
Take a look at your dataset and try to see which features are most predictive for job satisfaction.

#### Documentation
Document: what were your findings? What variables did you choose to focus on in the dataset? Do you think that your dataset is of good quality, or was it missing key data (demographic data, job values, etc.). Once the report is done, upload your notebook linked to this issue.

### Missing ONET_SOC_CODE values

In [ ]:
import pandas as pd

df = pd.read_csv('../../data/constructed_datasets/gss.csv')

miss = df['ONET_SOC_CODE'].isna()
print(f"Missing ONET_SOC_CODE: {miss.sum()} of {len(df)} rows ({miss.mean()*100:.2f}%)")

# Is missingness driven by OCC10 being missing, or by a crosswalk gap?
print("OCC10 missing:", df['OCC10'].isna().sum())
print("OCC10 present but ONET missing (crosswalk gap):", ((~df['OCC10'].isna()) & miss).sum())

# Who are the crosswalk-gap cases? - is missing ONET being caused by missing OCC10(native to gss)
gap = df[(~df['OCC10'].isna()) & miss]
print(gap['OCC10'].value_counts())

# Work status among all ONET-missing rows
print(df[miss]['WRKSTAT'].value_counts().sort_index())

Missing ONET_SOC_CODE: 5616 of 75699 rows (7.42%)
OCC10 missing: 5337
OCC10 present but ONET missing (crosswalk gap): 279
OCC10
9830.0    279
Name: count, dtype: int64
WRKSTAT
1.0     870
2.0     136
3.0      49
4.0     136
5.0     569
6.0     938
7.0    2586
8.0     290
Name: count, dtype: int64


**Summary:**

- 5,616 of 75,699 rows (~7.4%) are missing `ONET_SOC_CODE`.
- Of those, 5,337 rows (95%) are missing because `OCC10` itself is null, which means the person who took the survey had no job.
- The remaining 279 rows have a valid `OCC10` but still fail to map to an O*NET code. Every
  one of these 279 rows carries the same `OCC10` value: `9830`, "Military-Specific
  Occupations." O*NET-SOC doesn't have miltitary roles listed.
- Overall, missingness in `ONET_SOC_CODE` is fully explained by two non-random causes: not
  being employed, and holding a military occupation with no civilian O*NET analogue. There's
  no evidence of unexplained or random missingness in this variable.
- **What we should do:** rows missing `ONET_SOC_CODE` for either reason should be excluded from
  person-job-fit modeling  

### Variables of Interest


In [3]:
groups = {
    'Job satisfaction (target)': ['SATJOB', 'JOBSAT', 'EXJOBSAT', 'SATJOBHV'],
    'Job values - QWL importance battery': ['SECJOB', 'HIINC', 'PROMOTN', 'LEISURE', 'INTJOB',
                                             'WRKINDP', 'HLPOTHS', 'HLPSOC', 'FLEXHRS'],
    'Job values - ranked battery': ['JOBKEEP', 'JOBRISE', 'JOBHONOR', 'JOBOFF', 'JOBINTER',
                                     'JOBINDEP', 'JOBRESP', 'JOBPEOP', 'JOBHELP', 'JOBSOC',
                                     'JOBACCMP', 'JOBSAFE', 'JOBMEANS'],
    'Job data not in O*NET': ['TIMEPDWK', 'EMPSELF', 'SMALLBIG', 'PRIVGOVT', 'WANTJOB1',
                              'FLEXHRS1', 'WKPERSNL', 'FAMORJOB', 'JOBORFAM', 'WRKEARN',
                              'WRKENJOY', 'ILIKEJOB'],
    'Demographics': ['AGE', 'EDUC', 'SEX', 'RACE', 'MARITAL', 'SIBS', 'INCOME', 'RELIG16', 'ETHNIC'],
    'Linking keys': ['OCC10', 'ONET_SOC_CODE', 'WRKSTAT', 'YEAR'],
}

for label, cols in groups.items():
    print(f'--- {label} ---')
    for c in cols:
        n = df[c].notna().sum()
        print(f'  {c:10s} non-null={n:6d} ({n/len(df)*100:5.1f}%)')
    print()

--- Job satisfaction (target) ---
  SATJOB     non-null= 54131 ( 71.5%)
  JOBSAT     non-null=  3656 (  4.8%)
  EXJOBSAT   non-null=   485 (  0.6%)
  SATJOBHV   non-null=   294 (  0.4%)

--- Job values - QWL importance battery ---
  SECJOB     non-null=  5579 (  7.4%)
  HIINC      non-null=  5552 (  7.3%)
  PROMOTN    non-null=  5557 (  7.3%)
  LEISURE    non-null=  1403 (  1.9%)
  INTJOB     non-null=  5563 (  7.3%)
  WRKINDP    non-null=  5549 (  7.3%)
  HLPOTHS    non-null=  5556 (  7.3%)
  HLPSOC     non-null=  5552 (  7.3%)
  FLEXHRS    non-null=  4069 (  5.4%)

--- Job values - ranked battery ---
  JOBKEEP    non-null=   918 (  1.2%)
  JOBRISE    non-null=   915 (  1.2%)
  JOBHONOR   non-null=   914 (  1.2%)
  JOBOFF     non-null=   915 (  1.2%)
  JOBINTER   non-null=   915 (  1.2%)
  JOBINDEP   non-null=   914 (  1.2%)
  JOBRESP    non-null=   912 (  1.2%)
  JOBPEOP    non-null=   914 (  1.2%)
  JOBHELP    non-null=   918 (  1.2%)
  JOBSOC     non-null=   914 (  1.2%)
  JOBACCMP

**Summary:**

- **Job satisfaction (target):** `SATJOB` is the primary target, with 71.5% coverage
- **Job values - QWL importance battery** (`SECJOB`, `HIINC`, `PROMOTN`, `INTJOB`, `WRKINDP`,
  `HLPOTHS`, `HLPSOC`): the most direct bridge to O*NET Work Values, but coverage is only
  ~7.3% each, since this battery was only fielded in specific years to a subsample.
  `LEISURE` and `FLEXHRS` are even sparser (1.9% and 5.4%).
- **Job values - ranked battery:** `JOBMEANS` (single most-important job factor) has solid
  coverage at 28.8%, but the 12-item full ranking (`JOBKEEP` … `JOBSAFE`) is extremely sparse
  at ~1.2% each — asked in a different, much smaller module.
- **Job data not in O*NET:** all of these (hours preference, self-employment/sector
  preference, work-family balance, intrinsic motivation) sit in the 1.8%–7.4% coverage range.
  They add context O*NET can't provide, but none are usable as a primary feature at scale.
- **Demographics:** all near-complete (87.7%–99.9%), confirming these are safe to use as
  controls across virtually the whole dataset.
- **Linking keys:** `OCC10`/`ONET_SOC_CODE` are 92.6–92.9% complete (consistent with the
  gaps documented in Part 1), and `WRKSTAT`/`YEAR` are essentially fully populated.
- **What this means:** there's a hard split in this dataset between two very different levels
  of completeness, with almost nothing in between. Demographics, `SATJOB`, `WRKSTAT`, and the
  O*NET join sit at 70–100% coverage, while essentially every job-values variable sits at
  0.4%–7.4% coverage. We can definitely use the GSS for job satisfaction, but to fill in the gap for job values 
  we can use the ONET. However, the ONET doesnt have independent opinions, only a single value.

### Missing Values and Outliers in Variables of Interest


In [7]:
retained = {
    'Target': ['SATJOB'],
    'Demographics': ['AGE', 'EDUC', 'SEX', 'RACE', 'MARITAL', 'SIBS', 'INCOME', 'RELIG16', 'ETHNIC'],
    'Linking keys': ['OCC10', 'ONET_SOC_CODE', 'WRKSTAT', 'YEAR']
}
for label, cols in retained.items():
    print(f'--- {label} ---')
    for c in cols:
        n = df[c].notna().sum()
        print(f'  {c:10s} non-null={n:6d} ({n/len(df)*100:5.1f}%)')
    print()

# Usable rows for the core model: target + demographics + linking keys
core_cols = ['SATJOB', 'AGE', 'EDUC', 'SEX', 'RACE', 'MARITAL', 'INCOME', 'WRKSTAT', 'OCC10', 'ONET_SOC_CODE']
print('Core model usable rows:', df[core_cols].notna().all(axis=1).sum(), '/', len(df))

# Outlier / top-coding check on the numeric demographics
print(df['SIBS'].value_counts().sort_index())
print(df['AGE'].value_counts().sort_index().tail(5))

--- Target ---
  SATJOB     non-null= 54131 ( 71.5%)

--- Demographics ---
  AGE        non-null= 74829 ( 98.9%)
  EDUC       non-null= 75413 ( 99.6%)
  SEX        non-null= 75568 ( 99.8%)
  RACE       non-null= 75527 ( 99.8%)
  MARITAL    non-null= 75633 ( 99.9%)
  SIBS       non-null= 73896 ( 97.6%)
  INCOME     non-null= 66370 ( 87.7%)
  RELIG16    non-null= 72049 ( 95.2%)
  ETHNIC     non-null= 60908 ( 80.5%)

--- Linking keys ---
  OCC10      non-null= 70362 ( 92.9%)
  ONET_SOC_CODE non-null= 70083 ( 92.6%)
  WRKSTAT    non-null= 75652 ( 99.9%)
  YEAR       non-null= 75699 (100.0%)

Core model usable rows: 45709 / 75699
SIBS
0.0     3689
1.0    12998
2.0    14250
3.0    11693
4.0     8471
5.0     6018
6.0    16777
Name: count, dtype: int64
AGE
85.0    236
86.0    222
87.0    161
88.0    133
89.0    437
Name: count, dtype: int64


In [10]:
# INCOME distribution and top-bracket check
print(df['INCOME'].value_counts().sort_index())
print()

INCOME
1.0       969
2.0      1460
3.0      1335
4.0      1207
5.0      1339
6.0      1263
7.0      1331
8.0      2319
9.0      7025
10.0     5404
11.0     5667
12.0    37051
Name: count, dtype: int64



**Summary:**

- The remaining variables of interest split
  into three tiers of completeness: `SATJOB` at 71.5%, demographics/linking keys at
  80.5%–100%
- **Core model usable rows:** requiring `SATJOB` + all demographics + `WRKSTAT` +
  `OCC10`/`ONET_SOC_CODE` together leaves **45,709 of 75,699 rows (60.4%)** — a good sample
  size for a demographic/occupation-based satisfaction model
- The income variable presents an outlier issue, since the GSS survey makes participants choose on a scale of 1-12. 12 in the income category represents 37k, so if someone made way more than 37k then we would not know their true income.

### Initial EDA

In [11]:
df['SAT_SCORE'] = 5 - df['SATJOB']  # flip so higher = more satisfied

# Correlation of continuous/ordinal variables with satisfaction
for c in ['AGE', 'EDUC', 'SIBS', 'INCOME', 'YEAR']:
    sub = df[['SAT_SCORE', c]].dropna()
    print(f'{c:10s} n={len(sub):6d} corr={sub["SAT_SCORE"].corr(sub[c]):.3f}')

print()
print(df.groupby('WRKSTAT')['SAT_SCORE'].agg(['mean', 'count']))
print()
print(df.groupby('MARITAL')['SAT_SCORE'].agg(['mean', 'count']))
print()
print(df.groupby('RACE')['SAT_SCORE'].agg(['mean', 'count']))

# Satisfaction by education level, bucketed into standard credential tiers
df['educ_bucket'] = pd.cut(df['EDUC'], bins=[-1, 11, 12, 15, 16, 20],
                            labels=['<HS', 'HS', 'some college', 'bachelors', 'grad+'])
print(df.groupby('educ_bucket', observed=True)['SAT_SCORE'].agg(['mean', 'count']))

# Occupations with the highest/lowest average satisfaction (min 100 respondents)
occ = df.groupby('OCC10')['SAT_SCORE'].agg(['mean', 'count'])
occ = occ[occ['count'] >= 100].sort_values('mean')
print('LOWEST:'); print(occ.head(10))
print('HIGHEST:'); print(occ.tail(10))

AGE        n= 53568 corr=0.149
EDUC       n= 53955 corr=0.055
SIBS       n= 53964 corr=-0.016
INCOME     n= 48605 corr=0.107
YEAR       n= 54131 corr=-0.003

             mean  count
WRKSTAT                 
1.0      3.334500  34299
2.0      3.255106   7197
3.0      3.281250   1440
4.0      2.933477   2315
5.0      3.666667      6
6.0      3.166667      6
7.0      3.251496   8859
8.0      2.666667      3

             mean  count
MARITAL                 
1.0      3.370270  29411
2.0      3.416129   2790
3.0      3.264323   7226
4.0      3.181342   1908
5.0      3.116494  12756

          mean  count
RACE                 
1.0   3.324190  42725
2.0   3.139163   7696
3.0   3.241149   3587
                  mean  count
educ_bucket                  
<HS           3.225570   9691
HS            3.275186  16269
some college  3.277484  13244
bachelors     3.337126   8086
grad+         3.403451   6665
LOWEST:
            mean  count
OCC10                  
9620.0  2.974182    581
8800.0  2.97692

**Summary:**

- **None of the retained variables are strongly predictive on their own.** The strongest
  individual correlation with satisfaction is `AGE` at r=0.149 (older respondents report
  somewhat higher satisfaction), followed by `INCOME` at r=0.107 (caveat: `INCOME` is the
  top-coded variable documented above, so this is likely understated for recent years) and
  `EDUC` at r=0.055. `SIBS` (-0.016) and `YEAR` (-0.003) show essentially no relationship —
  notably, satisfaction hasn't trended up or down over the full 1972–2024 span.
- **Employment status matters more than any single continuous variable.** Respondents on
  temporary layoff/unemployed (WRKSTAT=4) report clearly lower satisfaction (mean 2.93 on
  the flipped 1–4 scale) than full-time workers (3.33) or any other status — consistent with
  job insecurity being a bigger driver than demographics.
- **Marital status shows a real gradient**: married (3.37) and widowed (3.42) respondents
  report the highest satisfaction, declining through divorced (3.26) and separated (3.18),
  down to never-married (3.12) — an 8% spread from top to bottom.
- **Education shows a clean, monotonic gradient**: satisfaction rises steadily from
  less-than-HS (3.23) through HS, some college, bachelor's, up to graduate+ (3.40) — the
  most consistent-looking pattern in this EDA, even though the correlation coefficient itself
  is modest.
- **Race shows a persistent gap**: Black respondents report lower satisfaction (3.14) than
  white (3.32) or other-race (3.24) respondents — the same direction as found in Part 2,
  now confirmed on the full retained sample rather than just the sparse job-values subset.

### Documentation


**Findings**

- `ONET_SOC_CODE` missingness (7.4%) is fully explained by two non-random causes: no
  occupation on record (respondent not in the labor force), and military occupations having
  no O*NET analogue. Neither needs imputation — both should simply be excluded.
- The GSS variables built to measure "job values" (the QWL importance battery and the ranked
  job-values battery) are too sparse to use — only 1.2%–7.4% of rows have them. O*NET
  can supply the job values, but it is one voice as opposed to multiple opinions.
- Two variables carry hidden data-quality problems that aren't visible from a simple
  missing-value count: `SIBS` and `AGE` are top-coded at their maximum values ("6 or more
  siblings," "89 or older"), and `INCOME` is much more seriously compromised — its top
  bracket is a fixed dollar max.
- In the initial EDA, no single retained variable strongly predicts job satisfaction — the
  best individual correlation was `AGE` at r=0.149. But categorical breakdowns showed
  meaningful, consistent gradients: employment status (unemployed respondents notably less
  satisfied), marital status (married/widowed higher than never-married), education (a clean
  monotonic increase), and race (Black respondents report lower satisfaction than white or
  other-race respondents). Occupation-level variation was the largest effect found — mean
  satisfaction spanned roughly 2.97 to 3.85 across occupations with sufficient sample size,
  a bigger spread than any demographic split produced on its own.

**What variables to focus on?**

The final retained set is: `SATJOB` (target), demographics (`AGE`, `EDUC`, `SEX`, `RACE`,
`MARITAL`, `SIBS`, `INCOME` with the caveat above, `RELIG16`, `ETHNIC`), and the linking keys
(`OCC10`, `ONET_SOC_CODE`, `WRKSTAT`, `YEAR`). This gives a 45,709-row usable core sample
(60.4% of the full file) — a large jump from the ~2,969 rows available if the sparse
GSS job-values variables were kept. The tradeoff: we gave up the ability to
compute an individual-level "stated values vs. job" fit score from GSS alone, in exchange for
a sample size large enough to support reliable modeling. The O*NET join (via
`ONET_SOC_CODE`) is expected to supply the job-characteristics side of "fit" going forward.

**Is the dataset good quality, or missing key data?**

Its neither good or bad. The core demographic/satisfaction/occupation data is high quality — well-covered,
consistent. But the dataset is missing the one thing
central to the project's stated goal: a usable, well-covered measure of individual job
values and actual job characteristics. The path forward relies on O*NET to supply job-side
characteristics at the occupation level, while accepting that "fit" will be measured against
occupation averages rather than truly individual stated preferences.



## Final Conclusion

#### GSS supplies job satisfaction and demographics matched with onet, but since ONET is not voicing indiviual opinions its less personalized. We would need a dataset that measures indiviual fit (ISSP) and matches with the GSS codes.

